In [1]:
%pip install pandas
print("panda is ready")


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
panda is ready


In [1]:
import numpy as np
import pandas as pd

print(
    "⏳ 正在執行 Transaction Banking Cash Management 數據處理與流動性分析..."
)

df = pd.read_csv("Fraud_transactions Data.csv")


df = df.dropna(subset=["amount", "type", "nameOrig"])
df["step"] = df["step"].astype(int)  # step 代表時間 (以小時為單位)


df["net_amount"] = np.where(
    df["type"] == "CASH_IN", df["amount"], -df["amount"]
)


df = df.sort_values(by=["nameOrig", "step"])

# 計算客戶交易後的即時賬戶餘額 (New Balance)
df["current_balance"] = df["newbalanceOrig"]

# 定義 Transaction Banking 的 Working Capital 門檻
MIN_SAFETY_BUFFER = 50000  


# 只要最新餘額低於安全門檻，且該筆交易係大額支出，即判定為 High Liquidity Risk
df["liquidity_risk_alert"] = (df["current_balance"] < MIN_SAFETY_BUFFER) & (
    df["net_amount"] < 0
)

# 計算被標記風險的客戶與交易總數
total_corporate_tx = len(df)
risk_alerts_count = df["liquidity_risk_alert"].sum()
risk_percentage = (risk_alerts_count / total_corporate_tx) * 100

print(f"📊 總企業交易筆數: {total_corporate_tx}")
print(f"⚠️ 觸發流動性預警 (Liquidity Risk Alert) 筆數: {risk_alerts_count}")
print(f"🔥 企業客戶營運資金風險比例: {risk_percentage:.2f}%\n")


df.to_csv("clean_tb_transactions.csv", index=False)
print("✅ Python 處理完成！已匯出為 clean_tb_transactions.csv")

⏳ 正在執行 Transaction Banking Cash Management 數據處理與流動性分析...
📊 總企業交易筆數: 6362620
⚠️ 觸發流動性預警 (Liquidity Risk Alert) 筆數: 4249124
🔥 企業客戶營運資金風險比例: 66.78%

✅ Python 處理完成！已匯出為 clean_tb_transactions.csv
